# 🦜🔗 LangChain para Dummies — Aprenda paso a paso

Bienvenido. Este cuaderno le enseña **LangChain desde cero**, con explicaciones sencillas, analogías y mucha práctica.

## ¿Cómo usar este cuaderno?
1. Ejecute las celdas **en orden**, de arriba hacia abajo (tecla `Shift + Enter`).
2. Lea cada explicación (celdas de texto) antes de ejecutar el código que le sigue.
3. Cuando vea **✏️ TU TURNO**, modifique el código y experimente: así se aprende de verdad.
4. Si algo falla, vaya a la sección **🚨 Errores comunes** al final.

## ¿Qué aprenderá?
| Lección | Tema | Lo que logrará |
|---|---|---|
| 0 | ¿Qué es LangChain? | Entender para qué sirve (sin tecnicismos) |
| 1 | Instalación y llaves API | Dejar todo listo |
| 2 | Su primer modelo | Hablar con un LLM desde Python |
| 3 | Plantillas de prompt | Reutilizar instrucciones con variables |
| 4 | Cadenas con LCEL (`\|`) | Conectar piezas como una línea de ensamblaje |
| 5 | Salidas estructuradas | Recibir datos ordenados (no texto libre) |
| 6 | Memoria de conversación | Que el chat recuerde lo anterior |
| 7 | RAG (buscar en sus documentos) | Preguntas sobre sus propios textos |
| 8 | Herramientas y agentes | Un asistente que decide y actúa solo |

> **Requisitos:** Python 3.10 o superior. No necesita saber nada de IA; basta Python básico (variables, funciones, listas).

## Lección 0 — ¿Qué es LangChain? (la explicación de la abuela)

Imagine que un modelo de lenguaje (como GPT) es un **cerebro muy listo encerrado en una caja**: sabe responder, pero no puede buscar en internet, ni leer sus archivos, ni recordar nada entre sesiones.

**LangChain es el cuerpo que le da a ese cerebro:**
- 🖐️ **Manos (herramientas):** para llamar APIs, calculadoras, bases de datos.
- 📚 **Biblioteca (RAG):** para consultar sus propios documentos antes de responder.
- 🧠 **Memoria:** para recordar la conversación.
- 🔗 **Cadenas (chains):** para conectar pasos uno tras otro, como una receta de cocina.

### La analogía del mesero
- El **modelo** es el cocinero: prepara el plato (genera texto).
- El **prompt** es la orden que usted escribe en la comanda.
- Una **cadena** es el mesero: toma su orden, la lleva a la cocina y le trae el plato servido.
- Un **agente** es un mesero con iniciativa: decide solo si necesita ir al mercado (buscar datos) antes de cocinar.

### ¿Qué NO es LangChain?
No es un modelo ni una IA nueva. Es una **librería de Python** que organiza y conecta modelos de distintos proveedores (OpenAI, Anthropic, Google, Ollama...) con una sola forma de trabajar.

## Lección 1 — Instalación y llaves API

### Instalación
Ejecute la siguiente celda **una sola vez**. Instalamos:
- `langchain` → el paquete principal (incluye agentes y las piezas básicas).
- `langchain[openai]` → el conector con OpenAI (puede cambiarlo por otro proveedor).

### La llave API 🔑
Una llave API es como la **contraseña de un club**: sin ella, el proveedor (OpenAI, por ejemplo) no le deja usar sus modelos. Reglas de oro:
1. **Nunca** la pegue directamente en el código.
2. Guárdela en una variable de entorno o escríbala cuando se la pidan.
3. Si no tiene llave, no se preocupe: este cuaderno incluye un **modo de práctica gratuito** con un modelo falso para que aprenda la mecánica.

In [ ]:
# ⬇️ Ejecute esta celda UNA vez para instalar LangChain (quite el "#" si falta algo)
# %pip install -qU langchain "langchain[openai]"
print("Si no vio errores, la instalación está lista ✅")

In [ ]:
# === CONFIGURACIÓN CENTRAL: edite SOLO esta celda según su caso ===
import os, getpass

# ¿Tiene llave de OpenAI? Ponga False para practicar gratis con un modelo falso.
MODO_PRACTICA = True   # ⬅️ cámbielo a False cuando tenga su API key

MODELO = "openai:gpt-5.5"   # formato "proveedor:modelo". Ejemplos:
# "anthropic:claude-sonnet-4-6" | "google_genai:gemini-2.5-flash-lite" | "ollama:llama3.2"

if not MODO_PRACTICA and not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Pegue su API key de OpenAI: ")

print("Modo práctica (gratis):", MODO_PRACTICA)

## Lección 2 — Su primer modelo: "Hola, mundo" con IA

Un **modelo de chat** recibe mensajes y devuelve un mensaje. Piense en los mensajes como un guion de teatro con tres papeles:
- **system** 🎬 (el director): define la personalidad y las reglas. El usuario nunca lo ve.
- **human** 🙋 (usted): lo que la persona pregunta.
- **ai** 🤖 (el actor): lo que el modelo responde.

La función `init_chat_model()` crea el modelo indicando `"proveedor:modelo"`, y **todos** los modelos de LangChain se usan igual: `.invoke()` (una respuesta), `.batch()` (varias a la vez) y `.stream()` (respuesta letra a letra).

In [ ]:
# Creamos el modelo (real o de práctica, según su configuración)
if MODO_PRACTICA:
    from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
    # Modelo FALSO de práctica: repite lo que usted dice. Cuesta $0.
    model = GenericFakeChatModel(messages=iter([
        "¡Hola! Soy un modelo de práctica. Cuando tenga una API key real, cambie MODO_PRACTICA a False.",
        "Entendido, sigo practicando.",
    ] * 50))
else:
    from langchain.chat_models import init_chat_model
    model = init_chat_model(MODELO, temperature=0)  # temperature=0 → respuestas predecibles

# Su primera llamada: invoke() envía y espera la respuesta completa
respuesta = model.invoke("¿Qué es LangChain? Explícalo en una frase.")
print(respuesta.content)

In [ ]:
# El mismo mensaje, pero usando el guion de teatro (system + human)
from langchain.messages import SystemMessage, HumanMessage

mensajes = [
    SystemMessage(content="Eres un profesor paciente que explica con analogías de cocina."),
    HumanMessage(content="¿Qué es una API?"),
]
respuesta = model.invoke(mensajes)
print(respuesta.content)

# ✏️ TU TURNO: cambie el system a "Eres un pirata gruñón" y vuelva a ejecutar.

### 🌡️ El parámetro `temperature` (el "nivel de creatividad")
- `temperature=0` → el modelo es un **contador serio**: siempre da la respuesta más probable. Ideal para datos y código.
- `temperature=1` o más → el modelo es un **poeta con cafeína**: respuestas variadas y sorprendentes. Ideal para ideas y redacción creativa.

## Lección 3 — Plantillas de prompt (el formulario reutilizable)

Escribir prompts a mano cada vez es como redactar el mismo correo mil veces. Una **plantilla** es un **formulario con espacios en blanco**: usted la diseña una vez y solo rellena los huecos.

Analogía: es una carta tipo *"Estimado {nombre}, su pedido {producto} llega el {fecha}"*. Las llaves `{}` son los huecos.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Definimos el formulario UNA vez, con dos huecos: {tema} y {nivel}
plantilla = ChatPromptTemplate.from_messages([
    ("system", "Eres un profesor que explica {tema} para un público de nivel {nivel}. Usa un ejemplo corto."),
    ("human", "Explícame: {pregunta}"),
])

# Rellenamos el formulario y vemos cómo queda el prompt final
prompt_listo = plantilla.invoke({
    "tema": "programación",
    "nivel": "principiante total",
    "pregunta": "¿qué es una variable?",
})
print(prompt_listo.to_messages())

## Lección 4 — Cadenas con LCEL: el operador `|` (línea de ensamblaje)

Aquí está la magia de LangChain. El operador **`|` conecta piezas** como tuberías: la salida de una pieza entra como entrada a la siguiente.

```
[formulario]  →  [cocinero]  →  [mesero que sirve el plato]
  plantilla   |    modelo    |    parser (extrae solo el texto)
```

- `plantilla | model` → el formulario rellenado entra al modelo.
- `... | StrOutputParser()` → en vez de un mensaje complejo, usted recibe **solo el texto**, limpio.

Toda pieza de LangChain obedece la misma interfaz: `.invoke()`, `.batch()`, `.stream()`. Aprenda una, aprendió todas.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# La línea de ensamblaje completa en UNA línea de código
cadena = plantilla | model | StrOutputParser()

# invoke(): una orden, una respuesta de texto limpio
resultado = cadena.invoke({
    "tema": "cocina",
    "nivel": "niño de 10 años",
    "pregunta": "¿por qué el agua hierve?",
})
print(resultado)

In [ ]:
# batch(): la MISMA cadena atiende varias órdenes a la vez (más rápido que un ciclo for)
preguntas = [
    {"tema": "música", "nivel": "adolescente", "pregunta": "¿qué es un compás?"},
    {"tema": "deporte", "nivel": "adulto", "pregunta": "¿qué es el fuera de juego?"},
]
for r in cadena.batch(preguntas):
    print("•", r, "\n")

# stream(): ve la respuesta aparecer pedazo a pedazo (como en ChatGPT)
# for trozo in cadena.stream({"tema": "cine", "nivel": "adulto", "pregunta": "¿qué es un plano secuencia?"}):
#     print(trozo, end="", flush=True)

# ✏️ TU TURNO: agregue una tercera pregunta a la lista y ejecute de nuevo.

## Lección 5 — Salidas estructuradas: datos ordenados, no texto libre

A veces no quiere un párrafo bonito: quiere **datos en casillas** (para guardarlos en Excel o una base de datos). Con `with_structured_output()` usted define un **formulario de salida** con Pydantic y el modelo está obligado a rellenarlo.

Analogía: es la diferencia entre pedir *"cuénteme del libro"* (texto libre) y entregar una **ficha de biblioteca** con campos: título, autor, nota del 1 al 10.

In [ ]:
from pydantic import BaseModel, Field

# 1. Diseñamos la ficha (cada campo tiene tipo y descripción)
class ResenaLibro(BaseModel):
    titulo: str = Field(description="Título del libro")
    autor: str = Field(description="Autor o autora")
    puntuacion: int = Field(description="Nota del 1 al 10")
    resumen: str = Field(description="Resumen en máximo 2 frases")

if MODO_PRACTICA:
    print("🔧 Modo práctica: la salida estructurada requiere un modelo real. Ejemplo de lo que obtendría:")
    print(ResenaLibro(titulo="Cien años de soledad", autor="Gabriel García Márquez",
                      puntuacion=10, resumen="La historia de la familia Buendía en Macondo. Un clásico del realismo mágico."))
else:
    # 2. Creamos un modelo que SOLO sabe rellenar esa ficha
    modelo_fichas = model.with_structured_output(ResenaLibro)
    ficha = modelo_fichas.invoke("Haz una reseña de 'Cien años de soledad' de García Márquez")
    print(type(ficha))          # <class 'ResenaLibro'> → ¡un objeto Python, no texto!
    print(ficha.puntuacion)     # accede a cada casilla por separado
    print(ficha.model_dump())   # o conviértalo a diccionario (listo para Excel/JSON)

# ✏️ TU TURNO: cree una clase 'RecetaCocina' con campos: nombre, ingredientes (list[str]), minutos (int).

## Lección 6 — Memoria: que el chat recuerde la conversación

Dato clave de novatos: **el modelo NO recuerda nada por sí solo**. Cada `invoke()` es como hablar con alguien que olvida todo al colgar el teléfono.

La solución "para dummies": guarde los mensajes en una **lista** y envíe el historial completo cada vez. Como un grupo de WhatsApp: el modelo lee todo el chat antes de contestar.

In [ ]:
from langchain.messages import AIMessage

# El historial es una simple lista de Python que crece con cada turno
historial = [SystemMessage(content="Eres un asistente amable y conciso. Responde en 1-2 frases.")]

def chatear(texto_usuario: str) -> str:
    historial.append(HumanMessage(content=texto_usuario))  # 1. añade su mensaje
    respuesta = model.invoke(historial)                    # 2. el modelo lee TODO el historial
    historial.append(respuesta)                            # 3. guarda la respuesta para el futuro
    return respuesta.content

print("🙋:", "Hola, me llamo Alejandro y entreno Muay Thai.")
print("🤖:", chatear("Hola, me llamo Alejandro y entreno Muay Thai."), "\n")
print("🙋:", "¿Cómo me llamo y qué deporte practico?")
print("🤖:", chatear("¿Cómo me llamo y qué deporte practico?"))
# Si el modelo responde correctamente la segunda pregunta, ¡la memoria funciona! 🎉

> **Ojo:** en proyectos grandes el historial se guarda con checkpointers (LangGraph) y se recorta para no gastar tokens de más. Para aprender, la lista es perfecta.

## Lección 7 — RAG: el examen a libro abierto 📖

Un LLM solo sabe lo que aprendió en su entrenamiento (y nada de sus documentos privados). **RAG** (Retrieval-Augmented Generation) es como permitirle un **examen a libro abierto**: antes de responder, busca el pasaje relevante en sus textos y contesta con eso delante.

### La tubería RAG en 5 pasos
1. **Cargar** documentos → 2. **Picarlos** en trozos (chunks) → 3. Convertir cada trozo en un **vector** (una huella numérica del significado) → 4. Guardarlos en un **almacén de vectores** → 5. Al preguntar, **buscar** los trozos más parecidos y dárselos al modelo como contexto.

¿Por qué vectores? Porque buscar por *significado* encuentra "canino" aunque usted escribió "perro". La búsqueda por palabra exacta fallaría.

In [ ]:
# --- RAG mínimo con sus propios textos ---
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

# 1. Sus "documentos" (aquí texto de ejemplo; en la vida real vienen de PDFs, Word, webs...)
textos = [
    Document(page_content="El club de lectura de Alejandro se reúne los jueves a las 7 p.m. en Bogotá. "
                          "Este mes leen 'La ciudad y los perros' de Mario Vargas Llosa."),
    Document(page_content="El entrenamiento de Muay Thai es lunes, miércoles y viernes a las 6 a.m. "
                          "Se debe llevar vendas de mano y protector bucal."),
    Document(page_content="La política de vacaciones de la empresa: 15 días hábiles al año, "
                          "se solicitan con 2 semanas de anticipación ante el jefe directo."),
]

# 2. Picar en trozos manejables (los textos largos se dividen con un poco de solape)
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
trozos = splitter.split_documents(textos)
print(f"Trozos creados: {len(trozos)}")

# 3 y 4. Convertir a vectores y guardar (los embeddings convierten texto → números)
if MODO_PRACTICA:
    from langchain_core.embeddings import DeterministicFakeEmbedding
    embeddings = DeterministicFakeEmbedding(size=128)   # gratis, para aprender la mecánica
else:
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings()

almacen = InMemoryVectorStore.from_documents(trozos, embeddings)
print("¡Base de conocimiento lista! ✅")

In [ ]:
# 5. Cadena RAG: buscar contexto → rellenar plantilla → modelo → texto
from langchain_core.runnables import RunnablePassthrough

buscador = almacen.as_retriever(k=2)   # trae los 2 trozos más relevantes

plantilla_rag = ChatPromptTemplate.from_messages([
    ("system", "Responde SOLO con la información del contexto. "
               "Si no está, di 'No tengo esa información'.\n\nContexto:\n{contexto}"),
    ("human", "{pregunta}"),
])

def formatear(docs):
    return "\n---\n".join(d.page_content for d in docs)

cadena_rag = (
    {"contexto": buscador | formatear, "pregunta": RunnablePassthrough()}  # busca y rellena
    | plantilla_rag
    | model
    | StrOutputParser()
)

print("P1:", cadena_rag.invoke("¿Cuándo y dónde se reúne el club de lectura?"), "\n")
print("P2:", cadena_rag.invoke("¿Qué debo llevar al entrenamiento?"), "\n")
print("P3:", cadena_rag.invoke("¿Cuál es la capital de Francia?"))  # trampa: no está en los docs

# ✏️ TU TURNO: agregue un Document con datos de su mascota y pregunte por ellos.

## Lección 8 — Herramientas y agentes: el asistente con caja de herramientas 🧰

Hasta ahora usted decidía el orden de los pasos. Un **agente** invierte los papeles: usted le da herramientas (funciones Python) y **el modelo decide cuándo usarlas**.

Analogía: un agente es un **empleado nuevo con una caja de herramientas**. Usted no le dice "usa el martillo ahora"; le describe el problema y él solo elige la herramienta correcta.

### Las 3 reglas de una buena herramienta
1. Es una función normal de Python con **tipos** (`ciudad: str`).
2. Tiene un **docstring** claro: el modelo lo lee para saber cuándo usarla. ¡El docstring es el manual de instrucciones!
3. Devuelve algo simple (texto o números).

En LangChain 1.x, `create_agent()` es la forma estándar de crear agentes: recibe el modelo, la lista de herramientas y un `system_prompt`.

In [ ]:
from langchain.tools import tool

@tool
def multiplicar(a: float, b: float) -> float:
    """Multiplica dos números. Úsala para cualquier cálculo de multiplicación."""
    return a * b

@tool
def obtener_clima(ciudad: str) -> str:
    """Obtiene el clima actual de una ciudad."""
    # En un proyecto real aquí llamaría a una API del clima. Para practicar, devolvemos un dato fijo.
    return f"En {ciudad} hace 18 °C y está nublado (dato de ejemplo)."

print("Herramientas listas:", [t.name for t in [multiplicar, obtener_clima]])

In [ ]:
from langchain.agents import create_agent

if MODO_PRACTICA:
    print("🔧 Modo práctica: el agente necesita un modelo real (que sepa decidir y llamar herramientas).")
    print("   Cuando tenga su API key, ponga MODO_PRACTICA = False y ejecute de nuevo.")
    print("   El código es idéntico: create_agent(modelo, tools=[...], system_prompt=...)")
else:
    agente = create_agent(
        model=MODELO,
        tools=[multiplicar, obtener_clima],
        system_prompt="Eres un asistente útil. Usa las herramientas cuando sea necesario.",
    )
    resultado = agente.invoke({"messages": [("user", "¿Cuánto es 37 por 4? Y dime el clima de Bogotá")]})
    # El último mensaje de la lista es la respuesta final del agente
    print(resultado["messages"][-1].content)

    # ¿Quiere ver qué hizo por dentro? Imprima todo el recorrido:
    # for m in resultado["messages"]:
    #     m.pretty_print()

# ✏️ TU TURNO: cree la herramienta 'calcular_imc(peso_kg, altura_m)' y agréguela a la lista.

## 🚨 Errores comunes (y su solución)

| Error típico | Por qué pasa | Solución |
|---|---|---|
| `AuthenticationError` / 401 | La API key falta o es inválida | Verifique la variable `OPENAI_API_KEY` y que la llave esté activa |
| `ModuleNotFoundError: langchain_openai` | Falta el paquete del proveedor | `pip install -U "langchain[openai]"` |
| Respuestas raras o inventadas en RAG | El contexto no tiene la respuesta | Es normal: el modelo debe decir "No tengo esa información" |
| El chat "olvida" todo | Cada `invoke()` es independiente | Envíe el historial completo (Lección 6) |
| Imports viejos de tutoriales (`LLMChain`, `langchain.chat_models.ChatOpenAI`) | Tutoriales de versiones antiguas | En LangChain 1.x use `init_chat_model`, cadenas con `\|` y `create_agent` |
| `RateLimitError` / 429 | Demasiadas peticiones o sin saldo | Espere, revise su plan de facturación del proveedor |

## 🎯 Mini-proyecto final
Combine todo en un **asistente de notas personales**:
1. Pase sus notas (texto) como `Document`s (Lección 7).
2. Cree una cadena RAG que responda solo con sus notas.
3. Añádale una herramienta `@tool` que guarde una nota nueva en una lista.
4. Conviértalo en agente con `create_agent` para que decida si debe buscar o guardar.
5. Extra: que las respuestas salgan como ficha Pydantic (Lección 5).

## 📚 Recursos oficiales para seguir
- Documentación: https://docs.langchain.com/oss/python/langchain/overview
- Quickstart de agentes: https://docs.langchain.com/oss/python/langchain/quickstart
- Guía de RAG/retrieval: https://docs.langchain.com/oss/python/langchain/retrieval
- Curso gratuito: https://academy.langchain.com/courses/langchain-essentials-python
- Referencia de la API: https://reference.langchain.com/python/

---
**¡Felicitaciones!** 🎉 Si completó las 8 lecciones, ya sabe más LangChain que el 90% de la gente que solo ha visto videos. El siguiente paso natural es **LangGraph** (flujos de agentes más complejos) y **LangSmith** (para depurar y medir sus cadenas).